Q1: A user's cohort is the month of their first order. For each cohort, return the number of users (users), the number of those users who ordered again in the following calendar month (retained), and retention_pct (retained / users × 100, rounded to 1 decimal). Cohort as text like 2022-01. Order by cohort.

In [72]:
import pandas as pd

orders = pd.DataFrame({
    "user_id": [1, 1, 1, 2, 2, 3, 3, 4, 5, 5, 6],
    "order_date": pd.to_datetime([
        "2022-01-05", "2022-02-10", "2022-04-01", "2022-01-20", "2022-03-02",
        "2022-02-03", "2022-02-25", "2022-02-14", "2022-03-07", "2022-04-09", "2022-03-30"]),
})

In [73]:
import numpy as np

orders['month_year'] = orders['order_date'].dt.strftime('%Y-%m')
orders['year'] = orders['order_date'].dt.year
orders['month']  = orders['order_date'].dt.month

orders['date'] = pd.to_datetime(orders[['year', 'month']].assign(day=1))

orders = orders.drop(columns='order_date')

orders_dedup = orders.drop_duplicates()

orders_dedup = orders_dedup.sort_values(['user_id', 'month_year'])

orders_dedup['lead_month_year'] = orders_dedup.groupby(['user_id'])['date'].shift(-1)

orders_dedup['order_rank'] = orders_dedup.groupby(['user_id'])['date'].rank(method='first', ascending=True)

orders_dedup_filtered = orders_dedup.loc[orders_dedup['order_rank'] == 1].copy()

lead_idx  = orders_dedup_filtered['lead_month_year'].dt.year * 12 + orders_dedup_filtered['lead_month_year'].dt.month
first_idx = orders_dedup_filtered['date'].dt.year * 12 + orders_dedup_filtered['date'].dt.month

orders_dedup_filtered['month_diff'] = lead_idx - first_idx

orders_dedup_filtered['retention_flag'] = np.where(orders_dedup_filtered['month_diff'] == 1, 1, 0)

agg_data = orders_dedup_filtered.groupby('month_year').agg(users = ('order_rank', 'sum'), retained = ('retention_flag', 'sum')).reset_index()

agg_data['retention_pct'] = (agg_data['retained']*100.0)/agg_data['users']

agg_data.rename(columns = {'month_year' : 'cohort'})


,cohort,users,retained,retention_pct
0,2022-01,2.0,1,50.0
1,2022-02,2.0,0,0.0
2,2022-03,2.0,1,50.0


Q2: Return every number that appears at least 3 times consecutively (in id order), with no repeats in the output. Return a single column consecutive_nums, ordered ascending.

In [114]:
import pandas as pd

logs = pd.DataFrame({
    "id":  range(1, 10),
    "num": [1, 1, 1, 2, 1, 2, 2, 2, 2],
})

In [115]:
logs = logs.sort_values('id')                 

logs['prev_num'] = logs['num'].shift(1)      

logs['new_start'] = np.where(logs['num'] != logs['prev_num'], 1, 0)

logs['islands'] = logs['new_start'].cumsum()

logs_agg = logs.groupby(['num', 'islands']).agg(reps = ('id', 'count')).reset_index()

logs_agg_filtered = logs_agg.loc[logs_agg['reps'] >= 3].reset_index()

result = logs_agg_filtered[['num']].rename(columns = {'num' : 'consecutive_nums'})

result

,consecutive_nums
0,1
1,2
